# B2 Pretraining, and what the model expects

Notebook B1 turned long tables into sequences of tokens. Now we train a transformer on them,
with exactly the task from notebook A1: **hide a token and ask the model what belongs in the
gap.**

In notebook A1 that meant

> The city of Oxford is located in `[MASK]`.

Here it means

> … `Prediabetes` · `Full-time employment` · `[MASK]` · `Medication review due` …

Same objective, same loss. Only the vocabulary changed.

> **IMPORTANT:** You are **not** training a model here. You will read the key parts of the training script, then load the model I trained and interrogate it. Training is the most time-consuming step in the whole pipeline.
> If you want the whole run rather than the highlights, it is in [`pretraining-the-full-model.ipynb`](pretraining-the-full-model.ipynb): an appendix notebook that I used to build and train the model end to end.

In [ ]:
# --- Setup: runs locally and on Colab --------------------------------
# On Colab this installs what is missing and pulls the model and the data from
# the Hugging Face Hub. In a local checkout it finds both in the repository and
# installs nothing. torch and numpy are left alone: Colab's builds are
# CUDA-matched, and replacing them costs minutes and forces a runtime restart.
import pathlib
import subprocess
import sys

try:
    import google.colab  # noqa: F401

    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "transformers==5.7.0"],
        check=True,
    )
    print("Colab: dependencies installed.")

import numpy as np
import pandas as pd
import torch

DATA_REPO = "carlomarxx/synthea-workshop-data"
MODEL_REPO = "carlomarxx/synthea-bert"


def resolve(local_path, repo_id, repo_type="dataset"):
    """Prefer the copy in this repository; fall back to the Hub if it is not here."""
    if pathlib.Path(local_path).exists():
        return str(local_path), "already in this repository"
    from huggingface_hub import snapshot_download

    return snapshot_download(repo_id, repo_type=repo_type), f"downloaded from {repo_id}"


DATA, data_source = resolve("../data/derived/workshop", DATA_REPO)
MODEL, model_source = resolve("../models/synthea-bert", MODEL_REPO, "model")

# event_bert.py sits beside the weights on the Hub, and in scripts/ locally.
sys.path[:0] = ["../scripts", MODEL]

print(f"data:  {DATA}\n       ({data_source})")
print(f"model: {MODEL}\n       ({model_source})")

# The tokenizer files are published beside the weights, so MODEL works on Colab.
TOKENIZER = (
    "../models/event-tokenizer"
    if pathlib.Path("../models/event-tokenizer").exists()
    else MODEL
)

vocabulary = pd.read_csv(f"{DATA}/vocabulary.csv")
sequences = pd.read_parquet(f"{DATA}/sequences.parquet")
cohort = pd.read_parquet(f"{DATA}/cohort.parquet").set_index("patient_id")

token_id = dict(zip(vocabulary.token, vocabulary.token_id))
name_of = dict(zip(vocabulary.token_id, vocabulary.token))
MASK, CLS, SEP = token_id["[MASK]"], token_id["[CLS]"], token_id["[SEP]"]
print(f"{len(vocabulary)} tokens, {len(sequences):,} sequences")


## 1. The tokenizer

A text tokenizer has to *discover* its vocabulary: it learns that "hospitalisation" is best
split into word pieces.

**Ours is the opposite:** the vocabulary is already fixed and finite, and every event is an
atom. So we use the simplest model HuggingFace's `tokenizers` library offers, `WordLevel`: a
lookup table from a string to an integer. The vocabulary is precomputed in
`data/derived/workshop/vocabulary.csv`.

The tokenizer and its metadata are stored in `models/event-tokenizer`.

In [ ]:
from transformers import PreTrainedTokenizerFast

tokenizer = PreTrainedTokenizerFast.from_pretrained(TOKENIZER)

demo = "\t".join(
    ["AGE_50_54", "SEX_F", "Sepsis (disorder)", "Full-time employment (finding)"]
)
encoded = tokenizer(demo)
print("tokens:", tokenizer.convert_ids_to_tokens(encoded["input_ids"]))
print("ids:   ", encoded["input_ids"])


# An event too rare to earn its own token falls back to [UNK].
rare = tokenizer("Meconium ileus (disorder)")
print("rare:  ", tokenizer.convert_ids_to_tokens(rare["input_ids"]))

## 2. How the model knows *when* things happened

Each position in the sequence is the sum of **four** vectors:

| | what it encodes | where it comes from |
|---|---|---|
| token embedding | **what** happened | learned, a 1,036 × 128 lookup table |
| absolute position | **order**: 1st event, 2nd, 3rd | BERT's own, added internally |
| Time2Vec(age) | **when** in the person's life | learned, the *life clock* |
| Time2Vec(days before index) | **how recently** | learned, the *recency clock* |

Absolute position gives order but not spacing: it cannot tell 3 days apart from 3 years
apart, because rank order throws the gaps away.

That matters here: a life is not evenly spaced, and "three hospital visits in a month" is a
very different thing from "three spread over twenty years".

[Time2Vec](https://arxiv.org/abs/1907.05321) (Kazemi et al., 2019) encodes a continuous time
`t` as one linear term plus a bank of sinusoids at *learned* frequencies:

$$ \text{t2v}(t)[0] = \omega_0 t + \varphi_0 \qquad
   \text{t2v}(t)[i] = \sin(\omega_i t + \varphi_i) \;\; (i > 0) $$

**Why two clocks?** They answer different questions. *Age* says "this happened at 52", which
is what a disease process cares about. *Days before the index date* says "this happened three
weeks ago", which is what a **prediction** cares about. A model given only age cannot tell a
recent crisis from an old one.

In [ ]:
from event_bert import Time2Vec

torch.manual_seed(0)
t2v = Time2Vec(out_dim=8)
ages = torch.linspace(35, 65, 300)
encoded = t2v(ages).detach()

import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(9, 3))
ax.plot(ages, encoded[:, 0], color="#1E152A", lw=2, label="linear term (trend)")
for i in range(1, 5):
    ax.plot(ages, encoded[:, i], lw=1, alpha=0.7, label=f"sinusoid {i}")
ax.set_xlabel("age in years")
ax.set_ylabel("value")
ax.set_title("Time2Vec: one trend term plus periodic terms at learned frequencies")
ax.legend(fontsize=8, frameon=False, ncol=5)
for s in ("top", "right"):
    ax.spines[s].set_visible(False)
plt.tight_layout()
plt.show()

## 3. (Extra) Two more architecture choices

`scripts/event_bert.py` differs from a stock BERT in two further ways, both following
[life2vec](https://doi.org/10.1038/s43588-023-00573-5). Each is a flag, so each can be
ablated.

### Tied input and output embeddings

The matrix that turns a token id into a vector and the matrix that turns a vector back into
a distribution over tokens are **the same matrix**. It halves the embedding parameters and
it is BERT's default (`tie_word_embeddings=True`).

### ReZero instead of LayerNorm

A transformer sublayer normally computes `LayerNorm(x + F(x))`.
[ReZero](https://arxiv.org/abs/2003.04887) computes

```
x + alpha * F(x)        with alpha initialised to 0
```

so at initialisation every sublayer is the identity and the signal passes through the stack
untouched. The network then learns how much of each sublayer it actually wants.

### (Extra) Performer attention, and when *not* to use it

Softmax attention is O(L²): every position attends to every other.
[Performer](https://arxiv.org/abs/2009.14794)'s FAVOR+ approximates the softmax kernel with
random features, which lets you re-associate the product and never build the L × L matrix at
all: O(L) instead.

life2vec uses Performer. We do not. Below is the timing comparison between standard (sdpa)
and Performer attention.

In [ ]:
# Measured on this machine by scripts/benchmark_attention.py.
benchmark = pd.DataFrame(
    {
        "sequence length": [128, 256, 512, 1024, 2048, 4096],
        "exact (sdpa) ms": [3.0, 3.7, 6.2, 18.6, 65.4, 252.2],
        "performer ms": [4.9, 5.9, 5.9, 11.0, 21.6, 41.6],
    }
)
benchmark["faster"] = np.where(
    benchmark["performer ms"] < benchmark["exact (sdpa) ms"], "performer", "exact"
)
benchmark

At **128 tokens (our sequence length) exact attention is both faster and not an
approximation.** Performer only starts winning past about 512 tokens, and by 4096 it is six
times faster and scaling linearly while exact attention scales quadratically.

> Approximate attention buys you sequence length.

life2vec needs it because a full life course from a national register runs to thousands of
events. Our five-year windows do not. If you extend this to whole lifetimes, switch it on
with `--attention performer` (the code is in `scripts/`).

## 4. The training script

Read [`scripts/pretrain.py`](../scripts/pretrain.py). Almost all of it is stock HuggingFace:

```python
config = BertConfig(vocab_size=1036, hidden_size=128, num_hidden_layers=4, ...)
model  = EventBertForMaskedLM(config)           # random weights

collator = DataCollatorForLanguageModeling(     # does the 30% masking, 80/10/10
    tokenizer=tokenizer, mlm=True, mlm_probability=0.30)

Trainer(model=model, args=TrainingArguments(...), data_collator=collator).train()
```

There is no hand-written training loop. `DataCollatorForLanguageModeling` is the same
collator you would use for English. It picks 30% of positions, replaces 80% of those with
`[MASK]`, 10% with a random token and leaves 10% as they are, and builds the labels.
Nothing in it is aware that the tokens are life events.

30% rather than BERT's original 15% follows [ModernBERT](https://arxiv.org/abs/2412.13663):
more masking is a harder task and a stronger signal per sequence, which matters on a 9 M
token corpus.

The only custom parts are the ones in sections 2 and 3: the two Time2Vec clocks added to the
token embeddings and handed to BERT as `inputs_embeds`, plus the ReZero residuals.

Training runs on **MPS** (Apple Silicon), CUDA, or CPU: `pretrain.py` picks whichever is
available. On an M-series laptop MPS is a bit over twice as fast as CPU.

In [ ]:
# The model: ~950K parameters. ModernBERT-base from notebook a1 has 149 MILLION.
from event_bert import EventBertForMaskedLM

model = EventBertForMaskedLM.from_pretrained(MODEL, expected_vocab_size=len(vocabulary))
n_params = sum(p.numel() for p in model.parameters())
print(
    f"{n_params:,} parameters, {model.config.num_hidden_layers} layers, "
    f"hidden size {model.config.hidden_size}"
)
print(
    f"attention: {model.attention}   ReZero: {model.rezero}   "
    f"tied embeddings: {model.embeddings_tied()}"
)

## 5. Asking the model to fill a gap

The function below builds a person's sequence, replaces one event with `[MASK]`, and reads
off the distribution over all 1,036 tokens at that position.

One thing to be careful about: the model was trained on token + position + **both Time2Vec
clocks**. If you feed it token embeddings alone it produces confident nonsense, so
`predict_masked` always passes `ages` and `days`.

In [ ]:
def build_input(row):
    """Compose [CLS] background [SEP] events [SEP], plus both clocks per position."""
    base_age = float(cohort.age_at_index.get(row.patient_id, 50.0))
    ids = [CLS, *row.background, SEP, *row.tokens, SEP]
    pad = len(row.background) + 2  # background and separators are timeless
    ages = (
        [base_age] * pad
        + [base_age - d / 365.25 for d in row.days_before_index]
        + [base_age]
    )
    days = [0.0] * pad + [float(d) for d in row.days_before_index] + [0.0]
    return ids, ages, days, pad


def predict_masked(row, position, k=5, also_mask=()):
    """Mask the event at `position` and return the model's top-k guesses for it.

    `also_mask` hides further event positions in the same forward pass but reports
    nothing about them. Use it to take away the context that would otherwise give
    the answer away, e.g. `also_mask=same_day(row, position)`.
    """
    ids, ages, days, offset = build_input(row)
    actual = name_of[ids[offset + position]]
    for hidden in {position, *also_mask}:
        ids[offset + hidden] = MASK

    with torch.no_grad():
        out = model(
            input_ids=torch.tensor([ids]),
            attention_mask=torch.ones(1, len(ids), dtype=torch.long),
            ages=torch.tensor([ages], dtype=torch.float),
            days=torch.tensor([days], dtype=torch.float),
        )
    probs = out.logits[0, offset + position].softmax(-1)
    top = probs.topk(k)
    return actual, [
        (name_of[int(i)], float(p)) for i, p in zip(top.indices, top.values)
    ]


def show(row, position):
    actual, guesses = predict_masked(row, position)
    print(f"actual: {actual}")
    for token, p in guesses:
        print(f"  {p:6.1%}  {token[:60]}{'   <-- correct' if token == actual else ''}")

In [ ]:
person = sequences.iloc[0]
print("background:", [name_of[t] for t in person.background])
print("\nfirst 12 events:")
for i, (tok, days) in enumerate(zip(person.tokens[:12], person.days_before_index[:12])):
    print(f"  [{i:2d}] {days:5d} days before index   {name_of[tok]}")

In [ ]:
show(person, 5)

### ✏️ Exercise 1: mask a position of your own

First, one patient printed, so there are positions to point at (use `EVENT_SPAN` to control
what to print).

In [ ]:
POSITION = 15  # <- change me
PERSON = 120  # <- and me

EVENT_SPAN = (10, 20)
person = sequences.iloc[PERSON]
# print("background:", [name_of[t] for t in person.background])


def show_person(row, first=0, last=None):
    """Print one patient the way the model receives them: background block, then events."""
    print(
        f"patient {row.patient_id}, {row.length} events, "
        f"age {cohort.age_at_index.get(row.patient_id, float('nan')):.0f} at the index date"
    )
    print("\nbackground (slot 0-5):")
    for slot, tok in enumerate(row.background):
        print(f"  <{slot}>  {name_of[tok]}")

    last = len(row.tokens) if last is None else last
    print(f"\nevents [{first}:{last}]  (events sharing a date are one encounter):")
    previous = None
    for i in range(first, min(last, len(row.tokens))):
        day = int(row.days_before_index[i])
        rule = "" if day == previous else "  ---"
        previous = day
        print(f"  [{i:2d}] {day:5d} days before index{rule}   {name_of[row.tokens[i]]}")


show_person(person, first=15, last=35)


Now mask some specific positions. Change `POSITION`, and try a different person too. For each
one, ask yourself:

- Are the top-5 guesses **plausible** given the surrounding events?
- Is the model confident (one token at 40%) or hedging (five tokens at 5% each)?
- When it is wrong, is it *sensibly* wrong (a different drug for the same condition), or is it
  nonsense?

A model that confidently predicts something absurd is a more useful teaching moment than a
clean success. If you find one, say so.

In [ ]:
print("\nWhat the model sees:")
for i, (tok, days) in enumerate(
    zip(
        person.tokens[EVENT_SPAN[0] : EVENT_SPAN[1]],
        person.days_before_index[EVENT_SPAN[0] : EVENT_SPAN[1]],
    )
):
    at = i + EVENT_SPAN[0]
    if at == POSITION:
        print(f"  [{at:2d}] {days:5d} days before index   [MASK]  <- {name_of[tok]}")
    else:
        print(f"  [{at:2d}] {days:5d} days before index   {name_of[tok]}")

print("\nPredictions:\n")

show(sequences.iloc[PERSON], POSITION)

### ✏️ Exercise 2: hide several things at once

`mask_and_predict` takes any mix of event positions and background slots, hides all of them
in one pass, and shows what the model puts back. Four things to try:

1. **One event from a bundle** -> say `events=[21]`. Easy, because its siblings are still there.
2. **A whole encounter** -> `events=same_day(person, 21)` hides every event on that date. Watch
   the confidence fall.
3. **A background slot** -> `background=[1]` hides `SEX`. Can the events give it back?
4. **A third of the timeline** -> `events=list(range(0, person.length, 3))`. This is what the
   training task looked like at 30% masking.

The `p(actual)` column is the one to read: the probability the model gave the true answer,
whether or not that was its top guess.

`k` sets how many guesses are shown per hidden slot. For a background slot that ranked list is
the interesting part: a masked `SEX` returns two candidates and a masked `INCOME` returns
twenty-five, so the shape of the list tells you how much the model knows.

In [ ]:
def same_day(row, position):
    """Every event position sharing a date with `position`, including it."""
    day = row.days_before_index[position]
    return [i for i in range(len(row.tokens)) if row.days_before_index[i] == day]


def mask_and_predict(row, events=(), background=(), k=3):
    """Hide any set of event positions and background slots, then read the model back."""
    ids, ages, days, offset = build_input(row)
    slots = [1 + b for b in background] + [offset + e for e in events]
    truth = {s: ids[s] for s in slots}
    for s in slots:
        ids[s] = MASK

    with torch.no_grad():
        out = model(
            input_ids=torch.tensor([ids]),
            attention_mask=torch.ones(1, len(ids), dtype=torch.long),
            ages=torch.tensor([ages], dtype=torch.float),
            days=torch.tensor([days], dtype=torch.float),
        )
    probs = out.logits[0].softmax(-1)

    rows = []
    for slot in sorted(slots):
        p = probs[slot]
        top = p.topk(3)
        guesses = [(name_of[int(i)], float(q)) for i, q in zip(top.indices, top.values)]
        rows.append(
            {
                "hid": f"<{slot - 1}> background"
                if slot < offset
                else f"[{slot - offset}] event",
                # Both clocks are pinned to the index date for background and separators.
                "days": int(days[slot]),
                "age": round(ages[slot], 1),
                "actual": name_of[truth[slot]][:40],
                f"model's top {k}": "  ".join(f"{t[:34]} {q:.2f}" for t, q in guesses),
                "p(actual)": round(float(p[truth[slot]]), 3),
                "correct": "yes" if guesses[0][0] == name_of[truth[slot]] else "NO",
            }
        )
    table = pd.DataFrame(rows)
    print(
        f"{len(table)} positions hidden, "
        f"{(table.correct == 'yes').mean():.0%} recovered exactly"
    )
    return table


pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 130)

# The default hides one whole encounter, plus SEX and INCOME from the background block.
# For one event only, use [21]; for a third of the timeline, range(0, person.length, 3).
FOCUS = min(
    21, len(person.tokens) - 1
)  # position 21 exists for the patient shipped here
EVENT_POSITIONS = same_day(person, FOCUS)  # (same day as event 21) <- change me
# EVENT_POSITIONS = [21]  # (just the focused event)
# EVENT_POSITIONS = list(range(0, len(person.tokens), 3))  # (every third event)
BACKGROUND_SLOTS = [1, 5]  # <- and me: 0 AGE, 1 SEX, 2 RACE, 3 ETH, 4 MARITAL, 5 INCOME

mask_and_predict(person, events=EVENT_POSITIONS, background=BACKGROUND_SLOTS)

Read the `model's top 3` column. Three things come out of the default run.

- **One event from an encounter is nearly free; the whole encounter is not.** Exercise 1 hid
  position 21 by itself and the model put 88% probability on the right answer. Hiding all
  eleven events of that day drops it to 0.24 on that same event, and its top guess is wrong
  at six of the eleven positions. The context that carries the answer is mostly the same day.
- **`SEX` comes back as `SEX_M 0.99` although it was hidden.** Nothing in the background block
  is left to say it, so the events said it instead. That is §7's point, and the reason
  deleting a protected attribute does not remove it from a model.
- **`INCOME` comes back as a flat list**: `INCOME_B24 0.05`, `INCOME_B23 0.05`,
  `INCOME_B22 0.05`. One in twenty across 25 bins is guessing. It ranked the true bin first
  here, and that is luck; on another patient it will not.

Try `events=list(range(0, person.length, 3))`. That hides a third of the timeline, roughly the
training task, and accuracy holds up because most events belong to protocols that repeat.

## 6. How good is it, really?

Top-5 accuracy over many random masked positions, against two reference points: guessing
uniformly at random, and always guessing the most frequent events in the corpus.

In [ ]:
rng = np.random.default_rng(0)
top1 = top5 = n = 0
for _ in range(200):
    row = sequences.iloc[rng.integers(len(sequences))]
    if len(row.tokens) < 5:
        continue
    actual, guesses = predict_masked(row, int(rng.integers(len(row.tokens))))
    tokens = [t for t, _ in guesses]
    top1 += tokens[0] == actual
    top5 += actual in tokens
    n += 1

commonest = (
    vocabulary[vocabulary.kind == "event"].nlargest(5, "frequency").token.tolist()
)
marginal = np.mean(
    [
        name_of[int(t)] in commonest
        for row in sequences.sample(200, random_state=0).itertuples()
        for t in [row.tokens[rng.integers(len(row.tokens))]]
    ]
)

print(f"model        top-1 {top1 / n:6.1%}   top-5 {top5 / n:6.1%}")
print(f"most-frequent-5 guess          top-5 {marginal:6.1%}")
print(f"uniform random                 top-5 {5 / len(vocabulary):6.2%}")

## 7. Does the model use the background block?

The background block says who the person is: age band, sex, race, ethnicity, marital status,
income bin (25 of them). Does any of it change what the model expects?

The rest of this section asks that question twice, in opposite directions. First we change the
background and watch the events. Then we hide the background and ask the events to give it
back. The two answers do not agree.

### ✏️ Exercise 3: swap a background token

Take one person, swap one background token, or several at once, and re-run the same masked
position.

In [ ]:
SLOTS = {"AGE": 0, "SEX": 1, "RACE": 2, "ETHNICITY": 3, "MARITAL": 4, "INCOME": 5}


def with_background(row, **changes):
    """Return a copy of `row` with one or more background tokens replaced.

    Slots are named, so several can change in one go:

        with_background(row, SEX="SEX_M")
        with_background(row, SEX="SEX_M", INCOME="INCOME_B01", AGE="AGE_30_34")

    Note the dict-style assignment below: `altered.background = ...` on a pandas Series
    sets an instance attribute and silently leaves the data untouched.
    """
    altered = row.copy()
    tokens = [name_of[t] for t in row.background]
    for slot, new_token in changes.items():
        if slot not in SLOTS:
            raise KeyError(f"unknown slot {slot!r}; pick one of {list(SLOTS)}")
        if new_token not in token_id:
            raise KeyError(f"{new_token!r} is not in the vocabulary")
        tokens[SLOTS[slot]] = new_token
    altered["background"] = np.array([token_id[t] for t in tokens], dtype=np.int16)
    return altered


def compare_backgrounds(row, position, *variants, k=3, also_mask=()):
    """Predict the same masked position under several background blocks.

    `also_mask` is passed straight through to `predict_masked`, so you can hide extra
    events at the same time and still read only `position`.
    """
    for changes in variants:
        altered = with_background(row, **changes)
        _, guesses = predict_masked(altered, position, also_mask=also_mask)
        label = ", ".join(f"{s}={v}" for s, v in changes.items()) or "unchanged"
        print(f"{label:<46}" + ",  ".join(f"{t[:28]} {p:.1%}" for t, p in guesses[:k]))


PERSON = 10
POSITION = 5  # example position to mask

row, position = sequences.iloc[PERSON], POSITION

print("Swapping only the SEX background slot")
compare_backgrounds(row, position, {}, {"SEX": "SEX_M"}, {"SEX": "SEX_F"})

print()
print("Swapping multiple background slots")
compare_backgrounds(
    row,
    position,
    {"SEX": "SEX_M", "AGE": "AGE_30_34", "INCOME": "INCOME_B01"},
    {"SEX": "SEX_F", "AGE": "AGE_55_59", "INCOME": "INCOME_B25"},
)

# Sanity check that the swap is actually reaching the model at all.
swapped = with_background(row, SEX="SEX_M", INCOME="INCOME_B01")


Try the same with `INCOME_B01` (poorest 4%) versus `INCOME_B25` (richest 4%), or a different
`AGE` band, or all of them at once. Two questions:

- Does the model's expectation shift at all?
- **Should** it? If income changes what the model expects to happen to someone medically,
  that is either a social gradient the simulator encoded, or a bias you have just taught it.

**If nothing moves, do not conclude anything yet.** The events around `position` may simply
be answering the question on their own: the whole encounter is still visible, and the
background block has nothing left to contribute. `also_mask` takes that context away. It
hides extra positions in the same forward pass and still reports only `position`. Hiding
the entire encounter is the harder test, and the fair one.

In [ ]:
# Take away the rest of the encounter, so the background block is all that is left to
# go on. `same_day` returns every position sharing a date with `position`.
encounter = same_day(row, position)
# encounter = list(range(0, len(person.tokens), 3))
print(f"hiding {len(encounter)} events from the same day, reading position {position}")

compare_backgrounds(
    row,
    position,
    {"AGE": "AGE_30_34", "SEX": "SEX_M", "MARITAL": "MARITAL_S"},
    {"AGE": "AGE_55_59", "SEX": "SEX_F", "MARITAL": "MARITAL_M"},
    also_mask=encounter,
)

One example proves nothing, so measure it. Below we flip each background attribute across
many people and positions, and record the **total variation distance** between the two
predicted distributions: 0 means the model ignored the attribute completely, 1 means it
changed its mind entirely.

As in the cell above, we hide the target's **whole day** rather than the single event, so the
rest of the encounter cannot answer for the model. That is the version of the probe to trust.
Leave the siblings visible and every distance is driven to zero by context the auditor never
meant to give away.

This is a counterfactual fairness probe, a standard auditing technique, and the same question
you would ask of a model trained on register data: *does this model's behaviour depend on a
protected attribute?*

In [ ]:
def distribution(row, position, background, also_mask=()):
    """Full predicted distribution at a masked position, for a given background block.

    `also_mask` hides further event positions in the same forward pass, so the
    encounter around `position` cannot answer the question on the model's behalf.
    """
    altered = row.copy()
    altered["background"] = np.array(background, dtype=np.int16)
    ids, ages, days, offset = build_input(altered)
    for hidden in {position, *also_mask}:
        ids[offset + hidden] = MASK
    with torch.no_grad():
        out = model(
            input_ids=torch.tensor([ids]),
            attention_mask=torch.ones(1, len(ids), dtype=torch.long),
            ages=torch.tensor([ages], dtype=torch.float),
            days=torch.tensor([days], dtype=torch.float),
        )
    return out.logits[0, offset + position].softmax(-1).numpy()


TESTS = {
    "AGE": (0, "AGE_35_39", "AGE_55_59"),
    "SEX": (1, "SEX_M", "SEX_F"),
    "RACE": (2, "RACE_white", "RACE_black"),
    "MARITAL": (4, "MARITAL_M", "MARITAL_S"),
    "INCOME": (5, "INCOME_B01", "INCOME_B25"),
}

rng = np.random.default_rng(0)
print("total variation distance when ONE background token is flipped,")
print("with the whole day around the target hidden")
print("(0 = the model ignores that attribute entirely)")
print()
for attribute, (slot, first, second) in TESTS.items():
    distances, flipped = [], 0
    for _ in range(60):  # number of people to sample
        person = sequences.iloc[rng.integers(len(sequences))]
        if len(person.tokens) < 5:
            continue
        where = int(rng.integers(len(person.tokens)))
        encounter = same_day(person, where)  # hide the whole day, not one event
        a, b = list(person.background), list(person.background)
        a[slot], b[slot] = token_id[first], token_id[second]
        pa = distribution(person, where, a, also_mask=encounter)
        pb = distribution(person, where, b, also_mask=encounter)
        distances.append(0.5 * np.abs(pa - pb).sum())
        flipped += int(pa.argmax() != pb.argmax())
    print(
        f"  {attribute:8s} mean {np.mean(distances):.4f}   max {np.max(distances):.4f}"
        f"   top-1 prediction changed in {flipped}/{len(distances)} cases"
    )

### Careful

The distances are small but no longer zero, and they order themselves sensibly: **SEX** moves
the distribution most (mean 0.062, and the top-1 prediction flips in 5 of 40 cases), then
**MARITAL** and **AGE**, with **RACE** and **INCOME** close to nothing. Hiding the whole
encounter multiplied every one of these by roughly five over the single-event version. Most of
what looked like "the model ignores the background" was context leaking in.

But even now, "the model barely uses the background block" does not follow. Look at what the
probe measures: the distribution at an **event** position. Two very different situations
produce the same small number:

1. the model learned that background is uninformative, or
2. the background embeddings never trained at all and are dead weights.

Nothing so far distinguishes them. So run a second probe: **mask a background token itself**
and see whether the model can recover it from the events.

In [ ]:
def recover_background(row, slot, k=3):
    """Mask a BACKGROUND token and ask the model to reconstruct it from the events."""
    ids, ages, days, _ = build_input(row)
    position = 1 + slot  # [CLS] first, then the background block
    actual = name_of[ids[position]]
    ids[position] = MASK
    with torch.no_grad():
        out = model(
            input_ids=torch.tensor([ids]),
            attention_mask=torch.ones(1, len(ids), dtype=torch.long),
            ages=torch.tensor([ages], dtype=torch.float),
            days=torch.tensor([days], dtype=torch.float),
        )
    probs = out.logits[0, position].softmax(-1).numpy()
    order = probs.argsort()[::-1][:k]
    return actual, [(name_of[int(i)], float(probs[i])) for i in order]


person = sequences.iloc[100]
print("events:", ", ".join(name_of[t] for t in person.tokens[:4]), "...")
for label, slot in [("SEX", 1), ("INCOME", 5)]:
    actual, guesses = recover_background(person, slot)
    print(f"\nmasked {label} (true value {actual}):")
    for token, p in guesses:
        print(f"    {p:6.1%}  {token}")

In [ ]:
# Across many people, against the base rate of always guessing the commonest value.
SLOTS = {"AGE": 0, "SEX": 1, "RACE": 2, "ETHNICITY": 3, "MARITAL": 4, "INCOME": 5}
background_values = pd.DataFrame(
    [[name_of[t] for t in r] for r in sequences.background], columns=list(SLOTS)
)

rng = np.random.default_rng(0)
rows = []
for attribute, slot in SLOTS.items():
    correct = n = 0
    for _ in range(100):
        person = sequences.iloc[rng.integers(len(sequences))]
        if len(person.tokens) < 5:
            continue
        actual, guesses = recover_background(person, slot)
        correct += guesses[0][0] == actual
        n += 1
    base = background_values[attribute].value_counts(normalize=True).iloc[0]
    rows.append(
        {
            "attribute": attribute,
            "values": background_values[attribute].nunique(),
            "base rate": base,
            "model recovers": correct / n,
            "lift (pp)": round((correct / n - base) * 100, 1),
        }
    )

pd.DataFrame(rows).sort_values("lift (pp)", ascending=False).set_index("attribute")

### What that actually shows

The embeddings are **alive**: the model puts essentially all of its probability mass on
background tokens when a background slot is masked, so it has learned the structure of the
sequence. But the recovery rates split hard in two:

- **Age and sex are both recovered at 99.0%**, far above their base rates (19.1% and 50.9%).
- **Race, ethnicity and income sit close to their base rates.** Income has 25 bins with a 4.1%
  base rate, and the model reaches 9.1%. Marital status is the middling case: 68.0% against a
  57.9% base rate.

**Question**: How does the model recover age and sex? And why does it fail with race and
income?

<details>
<summary>
<b>Answer</b>
</summary>

On age and sex: they are written all through the event stream, in antenatal procedures,
prostate screening and age-gated protocols.

On race and income: the model cannot infer these, because the events do not encode them.

So the right conclusion is not "the model ignores the background". It is:

> Information flows **events → background**, not the other way. Flipping the age token
> changes little because the events already say how old the person is. Flipping the income
> token changes nothing because *nothing* in the data says anything about income.

### The fairness point

> **You cannot remove a protected attribute from a model by deleting its token.**
> Delete `SEX` and the model reconstructs it from the events at 99.0%.

That is the proxy-variable problem, and it is the most transferable result in this notebook.
Fairness-by-deletion does not work, and here you can measure exactly how badly it fails.

One caveat, in the other direction: the *absence* of a race or income signal is a property of
**Synthea**, whose modules do not encode social gradients in care. On real register data that
income row would very likely not read +5.0 points. Running this exact probe is how you would
find out.

</details>

## 8. A caution before we go further

The model has learned *something*: it beats both baselines. But look at what it learned from.
Synthea is a rule-based simulator: a patient's events are generated by independent modules with
explicit conditions. Any structure the model recovers is the structure of **those rules**, not
of human illness.

The pipeline is real, and it is the same one you would run on register data. But it is a
reason to be careful about what we claim in the next notebook, when we start predicting.

Next: notebook B3 looks at the embedding space the model built, and notebook B4 uses it to
predict severe-disease onset.